# GSMThermodynBox2 Round-Trip Verification - Compact Demo

Quick verification of all Legendre transformations in GSMThermodynBox2.
Tests elastic-damage model with thermal capacity: F(T,ε,ω) = ½(1-ω)Eε² + C_ε(T - T*log(T/T_0))

**Primary criterion:** All round-trips F→X→F should return exact original expression.

In [2]:
# Global imports and setup
import sympy as sp
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box import GSMThermodynBox
from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import GSMStateFn, StateFunction
from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar

sp.init_printing()
print("GSMThermodynBox2 Round-Trip Verification")
print("=" * 50)

GSMThermodynBox2 Round-Trip Verification


In [10]:
# 1. SETUP: Define symbols and model

# Thermal variables
T = Scalar(r'\vartheta', codename='T', real=True, positive=True)
S = Scalar('S', codename='S', real=True)

# Mechanical variables  
eps = Scalar(r'\varepsilon', codename='eps', real=True)
sig = Scalar(r'\sigma', codename='sig', real=True)

# Damage variables
omega = Scalar(r'\omega', codename='omega', real=True, positive=True)
Y = Scalar('Y', codename='Y', real=True)

# Material parameters
E = Scalar('E', codename='E', positive=True)
C_eps = Scalar(r'C_{\varepsilon}', codename='C_eps', real=True, nonnegative=True)
T_0 = Scalar(r'\vartheta_0', codename='T_0', real=True, nonnegative=True)

# Extended Helmholtz free energy: elastic-damage + thermal capacity
F_elastic = sp.Rational(1, 2) * (1 - omega) * E * eps**2
F_thermal = C_eps * (T - T * sp.log(T / T_0))
F_original = F_elastic + F_thermal

print(f"Test Model: F(T,ε,ω) = {F_original}")

# Create GSMThermodynBox2
F_state_fn = GSMStateFn(F_original, T, S, eps, sig, omega, Y, StateFunction.HELMHOLTZ)
box = GSMThermodynBox(StateFunction.HELMHOLTZ, F_state_fn)

print("✓ GSMThermodynBox2 initialized with thermal elastic-damage model")

Test Model: F(T,ε,ω) = C_{\varepsilon}*(-\vartheta*log(\vartheta/\vartheta_0) + \vartheta) + E*\varepsilon**2*(1/2 - \omega/2)
✓ GSMThermodynBox2 initialized with thermal elastic-damage model


In [11]:
# 2. PROPERTY ACCESS: Get all state functions 

print("2. PROPERTY ACCESS:")
print("-" * 20)
print("Note: Validation warnings should now be resolved with enhanced symbol recognition")

F_expr = box.F.fn_expr
G_expr = box.G.fn_expr  
U_expr = box.U.fn_expr
H_expr = box.H.fn_expr

print(f"F: {F_expr}")
print(f"G: {G_expr}")
print(f"U: {U_expr}") 
print(f"H: {H_expr}")

2. PROPERTY ACCESS:
--------------------
Note: Validation warnings should now be resolved with enhanced symbol recognition
F: C_{\varepsilon}*(-\vartheta*log(\vartheta/\vartheta_0) + \vartheta) + E*\varepsilon**2*(1/2 - \omega/2)
G: (-C_{\varepsilon}*E*\vartheta*(\omega - 1)*(log(\vartheta/\vartheta_0) - 1) + \sigma**2/2)/(E*(\omega - 1))
U: -C_{\varepsilon}*\vartheta_0*(log(exp(S/C_{\varepsilon})) - 1)*exp(S/C_{\varepsilon}) - E*\varepsilon**2*(\omega - 1)/2 + S*\vartheta_0*exp(S/C_{\varepsilon})
H: (E*\vartheta_0*(\omega - 1)*(-C_{\varepsilon}*(log(exp(S/C_{\varepsilon})) - 1) + S)*exp(S/C_{\varepsilon}) + \sigma**2/2)/(E*(\omega - 1))


In [12]:
# 3. ROUND-TRIP TESTS: Primary verification criterion

print("3. ROUND-TRIP CONSISTENCY TESTS:")
print("-" * 35)

# Test F → G → F
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)
G_from_F = box.legendre_transform(StateFunction.GIBBS)
box.set_state_function(StateFunction.GIBBS, G_from_F)
F_from_G = box.legendre_transform(StateFunction.HELMHOLTZ)
FGF_diff = sp.simplify(F_from_G.fn_expr - F_original)
FGF_pass = FGF_diff == 0

print(f"F→G→F: {'✓ PASS' if FGF_pass else '✗ FAIL'} (diff: {FGF_diff})")

# Test F → U → F  
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)
U_from_F = box.legendre_transform(StateFunction.INTERNAL_ENERGY)
box.set_state_function(StateFunction.INTERNAL_ENERGY, U_from_F)
F_from_U = box.legendre_transform(StateFunction.HELMHOLTZ)
FUF_diff = sp.simplify(F_from_U.fn_expr - F_original)
FUF_pass = FUF_diff == 0

print(f"F→U→F: {'✓ PASS' if FUF_pass else '✗ FAIL'} (diff: {FUF_diff})")

# Test F → H → F
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)  
H_from_F = box.legendre_transform(StateFunction.ENTHALPY)
box.set_state_function(StateFunction.ENTHALPY, H_from_F)
F_from_H = box.legendre_transform(StateFunction.HELMHOLTZ)
FHF_diff = sp.simplify(F_from_H.fn_expr - F_original)
FHF_pass = FHF_diff == 0

print(f"F→H→F: {'✓ PASS' if FHF_pass else '✗ FAIL'} (diff: {FHF_diff})")

3. ROUND-TRIP CONSISTENCY TESTS:
-----------------------------------
F→G→F: ✓ PASS (diff: 0)
F→U→F: ✓ PASS (diff: 0)
F→H→F: ✓ PASS (diff: 0)


In [13]:
# Continue round-trip tests: G→H→G and U→G→U

# Test G → H → G
G_state_fn = GSMStateFn(G_expr, T, S, sig, eps, omega, Y, StateFunction.GIBBS)
box.set_state_function(StateFunction.GIBBS, G_state_fn)
H_from_G = box.legendre_transform(StateFunction.ENTHALPY)
box.set_state_function(StateFunction.ENTHALPY, H_from_G)
G_from_H = box.legendre_transform(StateFunction.GIBBS)
GHG_diff = sp.simplify(G_from_H.fn_expr - G_expr)
GHG_pass = GHG_diff == 0

print(f"G→H→G: {'✓ PASS' if GHG_pass else '✗ FAIL'} (diff: {GHG_diff})")

# Test U → G → U
U_state_fn = GSMStateFn(U_expr, S, T, eps, sig, omega, Y, StateFunction.INTERNAL_ENERGY)
box.set_state_function(StateFunction.INTERNAL_ENERGY, U_state_fn)
G_from_U = box.legendre_transform(StateFunction.GIBBS)
box.set_state_function(StateFunction.GIBBS, G_from_U)
U_from_G = box.legendre_transform(StateFunction.INTERNAL_ENERGY)
UGU_diff = sp.simplify(U_from_G.fn_expr - U_expr)
UGU_pass = UGU_diff == 0

print(f"U→G→U: {'✓ PASS' if UGU_pass else '✗ FAIL'} (diff: {UGU_diff})")

G→H→G: ✓ PASS (diff: 0)
U→G→U: ✓ PASS (diff: 0)


In [14]:
# 4. CONSTITUTIVE RELATIONS: Verify physical consistency

print("4. CONSTITUTIVE RELATIONS:")
print("-" * 27)

# Reset to original F for consistent derivatives
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)

# Extract stress from F and strain from G
sigma_from_F = sp.diff(F_expr, eps)  # σ = ∂F/∂ε
epsilon_from_G = -sp.diff(G_expr, sig)  # ε = -∂G/∂σ

print(f"σ = ∂F/∂ε = {sigma_from_F}")
print(f"ε = -∂G/∂σ = {epsilon_from_G}")

# Test inverse relationship: ε(σ(ε)) = ε
epsilon_test = epsilon_from_G.subs(sig, sigma_from_F)
inverse_correct = sp.simplify(epsilon_test - eps) == 0

print(f"Inverse test ε(σ(ε)) = ε: {'✓ CORRECT' if inverse_correct else '✗ INCORRECT'}")

# Damage force consistency
Y_from_F = -sp.diff(F_expr, omega)  # Y = -∂F/∂ω
print(f"Y = -∂F/∂ω = {Y_from_F}")

# Entropy from F and U
S_from_F = -sp.diff(F_expr, T)  # S = -∂F/∂T
T_from_U = sp.diff(U_expr, S)  # T = ∂U/∂S
print(f"S = -∂F/∂T = {S_from_F}")
print(f"T = ∂U/∂S = {T_from_U}")

4. CONSTITUTIVE RELATIONS:
---------------------------
σ = ∂F/∂ε = 2*E*\varepsilon*(1/2 - \omega/2)
ε = -∂G/∂σ = -\sigma/(E*(\omega - 1))
Inverse test ε(σ(ε)) = ε: ✓ CORRECT
Y = -∂F/∂ω = E*\varepsilon**2/2
S = -∂F/∂T = C_{\varepsilon}*log(\vartheta/\vartheta_0)
T = ∂U/∂S = -\vartheta_0*(log(exp(S/C_{\varepsilon})) - 1)*exp(S/C_{\varepsilon}) + S*\vartheta_0*exp(S/C_{\varepsilon})/C_{\varepsilon}


In [15]:
# 5. SUMMARY AND FINAL STATE FUNCTIONS

all_round_trips = [FGF_pass, FUF_pass, FHF_pass, GHG_pass, UGU_pass]
all_pass = all(all_round_trips)

print("=" * 50)
print("VERIFICATION SUMMARY:")
print("-" * 20)
print(f"Round-trip tests: {sum(all_round_trips)}/5 pass")
print(f"Constitutive relations: {'✓' if inverse_correct else '✗'}")

if all_pass:
    print("\n🎉 ALL ROUND-TRIPS PASS - IMPLEMENTATION VERIFIED!")
    print("GSMThermodynBox2 correctly implements Legendre transformations")
    print("with constitutive relation substitutions.")
else:
    print("\n⚠ SOME TESTS FAILED - REVIEW NEEDED")
    failed_tests = [name for name, passed in 
                   zip(['F→G→F', 'F→U→F', 'F→H→F', 'G→H→G', 'U→G→U'], all_round_trips) 
                   if not passed]
    print(f"Failed tests: {failed_tests}")

print("=" * 50)

# Final state functions display
print("\nFINAL STATE FUNCTIONS:")
print("-" * 26)

# Reset and get final expressions via properties
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)
final_F = box.F.fn_expr
final_G = box.G.fn_expr  
final_U = box.U.fn_expr
final_H = box.H.fn_expr

print(f"F(T,ε,ω) = {final_F}")
print(f"G(T,σ,ω) = {final_G}")
print(f"U(S,ε,ω) = {final_U}")
print(f"H(S,σ,ω) = {final_H}")

print(f"\n✓ Verification complete. Primary criterion: {'PASSED' if all_pass else 'FAILED'}")
print(f"✓ Thermal model with elastic-damage successfully verified")
print(f"✓ All {len(all_round_trips)} round-trip transformations tested")

VERIFICATION SUMMARY:
--------------------
Round-trip tests: 5/5 pass
Constitutive relations: ✓

🎉 ALL ROUND-TRIPS PASS - IMPLEMENTATION VERIFIED!
GSMThermodynBox2 correctly implements Legendre transformations
with constitutive relation substitutions.

FINAL STATE FUNCTIONS:
--------------------------
F(T,ε,ω) = C_{\varepsilon}*(-\vartheta*log(\vartheta/\vartheta_0) + \vartheta) + E*\varepsilon**2*(1/2 - \omega/2)
G(T,σ,ω) = (-C_{\varepsilon}*E*\vartheta*(\omega - 1)*(log(\vartheta/\vartheta_0) - 1) + \sigma**2/2)/(E*(\omega - 1))
U(S,ε,ω) = -C_{\varepsilon}*\vartheta_0*(log(exp(S/C_{\varepsilon})) - 1)*exp(S/C_{\varepsilon}) - E*\varepsilon**2*(\omega - 1)/2 + S*\vartheta_0*exp(S/C_{\varepsilon})
H(S,σ,ω) = (E*\vartheta_0*(\omega - 1)*(-C_{\varepsilon}*(log(exp(S/C_{\varepsilon})) - 1) + S)*exp(S/C_{\varepsilon}) + \sigma**2/2)/(E*(\omega - 1))

✓ Verification complete. Primary criterion: PASSED
✓ Thermal model with elastic-damage successfully verified
✓ All 5 round-trip transformatio

In [16]:
# 6. PATH INDEPENDENCE: Demonstrate F→G→H = F→U→H

from IPython.display import display, Math

print("6. PATH INDEPENDENCE VERIFICATION:")
print("-" * 37)
print("Testing that different transformation paths to the same target give identical results")

# Reset to original F
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)

# Path 1: F → G → H
print("\nPath 1: F → G → H")
G_via_F = box.legendre_transform(StateFunction.GIBBS)
box.set_state_function(StateFunction.GIBBS, G_via_F)
H_via_FG = box.legendre_transform(StateFunction.ENTHALPY)

print("F → G transformation:")
display(Math(f"G = {sp.latex(G_via_F.fn_expr)}"))
print("G → H transformation:")
display(Math(f"H_{{FGH}} = {sp.latex(H_via_FG.fn_expr)}"))

# Path 2: F → U → H
print("\nPath 2: F → U → H")
box.set_state_function(StateFunction.HELMHOLTZ, F_state_fn)  # Reset
U_via_F = box.legendre_transform(StateFunction.INTERNAL_ENERGY)
box.set_state_function(StateFunction.INTERNAL_ENERGY, U_via_F)
H_via_FU = box.legendre_transform(StateFunction.ENTHALPY)

print("F → U transformation:")
display(Math(f"U = {sp.latex(U_via_F.fn_expr)}"))
print("U → H transformation:")
display(Math(f"H_{{FUH}} = {sp.latex(H_via_FU.fn_expr)}"))

# Compare the final H expressions
H_difference = sp.simplify(H_via_FG.fn_expr - H_via_FU.fn_expr)
paths_identical = H_difference == 0

print("\nPath Independence Test:")
print("Comparing final H expressions:")
display(Math(f"H \\text{{ via }} F \\to G \\to H = {sp.latex(H_via_FG.fn_expr)}"))
display(Math(f"H \\text{{ via }} F \\to U \\to H = {sp.latex(H_via_FU.fn_expr)}"))
display(Math(f"\\text{{Difference}} = {sp.latex(H_difference)}"))

print(f"Paths identical: {'✓ YES' if paths_identical else '✗ NO'}")

if paths_identical:
    print("\n🎉 PATH INDEPENDENCE VERIFIED!")
    print("Different transformation paths to the same state function yield identical results.")
    print("This confirms the mathematical consistency of Legendre transformations.")
else:
    print("\n⚠ PATH DEPENDENCE DETECTED - POTENTIAL ISSUE")
    
print("\nThis demonstrates that the thermodynamic state function graph is well-defined:")
print("• F → G → H ≡ F → U → H (both paths lead to the same H)")
print("• Legendre transformations form a consistent mathematical framework")
print("• State functions are path-independent thermodynamic potentials")

6. PATH INDEPENDENCE VERIFICATION:
-------------------------------------
Testing that different transformation paths to the same target give identical results

Path 1: F → G → H
F → G transformation:


<IPython.core.display.Math object>

G → H transformation:


<IPython.core.display.Math object>


Path 2: F → U → H
F → U transformation:


...


Path Independence Test:
Comparing final H expressions:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Paths identical: ✓ YES

🎉 PATH INDEPENDENCE VERIFIED!
Different transformation paths to the same state function yield identical results.
This confirms the mathematical consistency of Legendre transformations.

This demonstrates that the thermodynamic state function graph is well-defined:
• F → G → H ≡ F → U → H (both paths lead to the same H)
• Legendre transformations form a consistent mathematical framework
• State functions are path-independent thermodynamic potentials
